# 15. V1(전주 라벨 × 핵심 피처 상호작용) 반영 앙상블 — submit_41
14에서 저장한 중간 결과(`_refine_cache.pkl`)를 불러와 학습·예측만 수행.
- V1: FC(=submit_23 피처) + 상호작용 16개, LR(C=0.01). 검증(전주 라벨 있는 행) 0.5752 → 0.5809
- submit_41 = 0.35·s4 + 0.35·s3 + 0.30·V1 (submit_24에서 submit_23 자리를 V1로 교체)

In [1]:
import pickle, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
d = pickle.load(open('../data/processed/_refine_cache.pkl', 'rb'))
po, test, FC, INTER = d['po'], d['test'], d['FC'], d['INTER']
logreg = lambda C: Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                             ('m', LogisticRegression(max_iter=3000, C=C, class_weight='balanced'))])
rank01 = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / len(x)
p_v1 = logreg(0.01).fit(po[FC + INTER], po.target).predict_proba(test[FC + INTER])[:, 1]

SUBD = '../submissions'
sample = pd.read_csv('../data/raw/epoch_data/sample_submission.csv', encoding='utf-8-sig')
ext = lambda f: pd.read_csv(f'{SUBD}/{f}').set_index('row_id').prediction.loc[test.row_id].values
r4, r3 = rank01(ext('submit_10_s4_07_defl.csv')), rank01(ext('submit_10_s3_cons_defl_adj.csv'))
score = 0.35 * r4 + 0.35 * r3 + 0.30 * rank01(p_v1) + 1e-6 * rank01(p_v1)
s = rankdata(score) / len(score); s = (s - s.min()) / (s.max() - s.min())
sub = sample[['row_id']].merge(pd.DataFrame({'row_id': test.row_id.values, 'prediction': s}), on='row_id', how='left')
assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
assert set(sub.row_id) == set(sample.row_id) and sub.prediction.between(0, 1).all() and sub.prediction.notna().all()
assert not sub.prediction.duplicated().any()
sub.to_csv(f'{SUBD}/submit_41_ens_s4_s3_v1inter.csv', index=False, encoding='utf-8')
s26 = sub.set_index('row_id').prediction
for f in ['submit_21_ens_s4_s3_s15.csv', 'submit_24_ens_s4_s3_s23.csv']:
    o = pd.read_csv(f'{SUBD}/{f}').set_index('row_id').prediction
    print(f, '와 순위상관', round(s26.corr(o.loc[s26.index], method='spearman'), 4))

submit_21_ens_s4_s3_s15.csv 와 순위상관 0.9931
submit_24_ens_s4_s3_s23.csv 와 순위상관 0.9977
